
# Homework 3 — Advanced experiments and model optimization

## 1. Experimental objectives

In this notebook, I investigate additional methods for improving multilingual AI-generated text detection beyond the classical baseline and fine-tuned XLM-RoBERTa encoder.

My experiments focus on three directions:

1. Probability-level ensembling of TF-IDF Logistic Regression and XLM-RoBERTa with head-tail truncation.
2. Class-specific decision calibration to optimize macro-F1, particularly for the challenging `fusion` class.
3. Stylometric feature engineering and its combination with classical text representations.

I use the same group-aware validation fold established during EDA. I compare each experiment with the fixed baseline and track class-specific metrics.

I also reserve part of the validation data for assessing selected ensemble and calibration parameters, reducing the risk of overly optimistic results from repeated tuning.

My goal is to identify improvements that generalize across languages, domains, and document lengths.


In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import f1_score, classification_report
from IPython.display import display
from google.colab import drive

SEED = 42
CLASS_ORDER = ["human", "ai", "fusion"]

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/UCU_AI_HW3")

XLMR_PROBS_PATH = (
    PROJECT_DIR / "xlmr_checkpoint2500_validation_probs.csv"
)

FOLDS_PATH = PROJECT_DIR / "cv_folds_v1.csv"

print("Project directory:", PROJECT_DIR)
print("XLM-R probabilities:", XLMR_PROBS_PATH.exists())
print("Validation folds:", FOLDS_PATH.exists())

print("\nAvailable model artifacts:")
for pattern in [
    "*validation*prob*",
    "*baseline*",
    "*tfidf*"
]:
    for path in sorted(PROJECT_DIR.glob(pattern)):
        print(" -", path.name)

Mounted at /content/drive
Project directory: /content/drive/MyDrive/UCU_AI_HW3
XLM-R probabilities: True
Validation folds: True

Available model artifacts:
 - xlmr_checkpoint2500_validation_probs.csv
 - baseline_parameter_search_v1.csv
 - submission_word_tfidf_lr_v1.csv


In [ ]:
assert XLMR_PROBS_PATH.exists(), (
    "Saved XLM-R validation probabilities are missing."
)

xlmr_df = pd.read_csv(
    XLMR_PROBS_PATH,
    dtype={"id": str}
)

required_columns = [
    "id",
    "true_label",
    "head_human",
    "head_ai",
    "head_fusion",
    "headtail_human",
    "headtail_ai",
    "headtail_fusion"
]

assert set(required_columns).issubset(xlmr_df.columns)
assert len(xlmr_df) == 55_031
assert xlmr_df["id"].is_unique
assert xlmr_df["true_label"].isin(CLASS_ORDER).all()

head_probs = xlmr_df[
    ["head_human", "head_ai", "head_fusion"]
].to_numpy(dtype=np.float64)

headtail_probs = xlmr_df[
    ["headtail_human", "headtail_ai", "headtail_fusion"]
].to_numpy(dtype=np.float64)

for name, probs in [
    ("Head-only", head_probs),
    ("Head-tail", headtail_probs)
]:
    assert np.isfinite(probs).all()
    assert (probs >= 0).all()
    assert np.allclose(probs.sum(axis=1), 1, atol=1e-4)

    predictions = np.array(CLASS_ORDER)[
        np.argmax(probs, axis=1)
    ]

    score = f1_score(
        xlmr_df["true_label"],
        predictions,
        labels=CLASS_ORDER,
        average="macro"
    )

    print(f"{name} validation macro-F1: {score:.5f}")

    if name == "Head-only":
        assert abs(score - 0.85850) < 0.0001
    else:
        assert abs(score - 0.87345) < 0.0001

print("\nProbability verification: PASSED")
print("Validation documents:", len(xlmr_df))

Head-only validation macro-F1: 0.85850
Head-tail validation macro-F1: 0.87345

Probability verification: PASSED
Validation documents: 55031



## 2. Restore the classical baseline for ensembling

I reproduce the word-level TF-IDF Logistic Regression baseline using the original group-aware cross-validation split.

I fit the vectorizer and classifier exclusively on training folds 1–4 and generate class probabilities for validation fold 0.

I verify that the reproduced model achieves the original validation macro-F1 of approximately 0.87852. I then align its predictions with the saved XLM-RoBERTa validation probabilities using document IDs.

This prevents data leakage and ensures that subsequent ensemble experiments compare the models on identical examples.


In [ ]:
import os
import zipfile

from pathlib import Path
from google.colab import userdata

COMPETITION = "ai-generated-text-detection-hw3-ucu-ai-2026"

DATA_DIR = Path("/content/hw3_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"

if not TRAIN_PATH.exists():
    os.environ["KAGGLE_API_TOKEN"] = (
        userdata.get("kaggle_token").strip()
    )

    !kaggle competitions download -c {COMPETITION} -p {DATA_DIR}

    for archive_path in DATA_DIR.glob("*.zip"):
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(DATA_DIR)

assert TRAIN_PATH.exists(), "Training dataset not found."

train_df = pd.read_csv(
    TRAIN_PATH,
    dtype={"id": str}
)

folds_df = pd.read_csv(
    FOLDS_PATH,
    dtype={"id": str}
)

train_df = train_df.merge(
    folds_df[["id", "cv_fold"]],
    on="id",
    how="left",
    validate="one_to_one"
)

assert len(train_df) == 272_352
assert train_df["cv_fold"].notna().all()

baseline_train_df = train_df[
    train_df["cv_fold"] != 0
].copy()

baseline_val_df = train_df[
    train_df["cv_fold"] == 0
].copy()

print("Training documents:", len(baseline_train_df))
print("Validation documents:", len(baseline_val_df))

assert len(baseline_train_df) == 217_321
assert len(baseline_val_df) == 55_031

print("Training data restored successfully.")

100% 380M/380M [00:21<00:00, 18.9MB/s]

Training documents: 217321
Validation documents: 55031
Training data restored successfully.


In [ ]:
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

SEED = 42
CLASS_ORDER = ["human", "ai", "fusion"]

baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.98,
            max_features=150_000,
            sublinear_tf=True,
            strip_accents=None,
            lowercase=True,
            dtype=np.float32
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            max_iter=300,
            solver="lbfgs",
            random_state=SEED
        )
    )
])

start_time = time.perf_counter()

baseline_model.fit(
    baseline_train_df["text"],
    baseline_train_df["label"]
)

elapsed = time.perf_counter() - start_time

baseline_val_probs = baseline_model.predict_proba(
    baseline_val_df["text"]
)

model_classes = baseline_model.named_steps[
    "classifier"
].classes_

class_indices = [
    list(model_classes).index(label)
    for label in CLASS_ORDER
]

baseline_val_probs = baseline_val_probs[:, class_indices]

baseline_val_preds = np.array(CLASS_ORDER)[
    np.argmax(baseline_val_probs, axis=1)
]

baseline_f1 = f1_score(
    baseline_val_df["label"],
    baseline_val_preds,
    labels=CLASS_ORDER,
    average="macro"
)

print(f"Training time: {elapsed / 60:.2f} minutes")
print(f"Validation macro-F1: {baseline_f1:.5f}")
print("Probability matrix:", baseline_val_probs.shape)

assert abs(baseline_f1 - 0.87852) < 0.0001, (
    "Baseline result differs from the original experiment. "
    "Check the configuration and validation split."
)

print("Baseline reproduction: PASSED")

Training time: 9.83 minutes
Validation macro-F1: 0.87852
Probability matrix: (55031, 3)
Baseline reproduction: PASSED


In [ ]:
baseline_probs_df = pd.DataFrame({
    "id": baseline_val_df["id"].astype(str).to_numpy(),
    "true_label": baseline_val_df["label"].to_numpy(),
    "tfidf_human": baseline_val_probs[:, 0],
    "tfidf_ai": baseline_val_probs[:, 1],
    "tfidf_fusion": baseline_val_probs[:, 2]
})

baseline_probs_path = (
    PROJECT_DIR / "tfidf_validation_probs_v1.csv"
)

baseline_probs_df.to_csv(
    baseline_probs_path,
    index=False
)

ensemble_df = xlmr_df.merge(
    baseline_probs_df,
    on="id",
    how="inner",
    validate="one_to_one",
    suffixes=("", "_tfidf")
)

assert len(ensemble_df) == 55_031

assert (
    ensemble_df["true_label"]
    == ensemble_df["true_label_tfidf"]
).all()

print("TF-IDF probabilities saved:", baseline_probs_path)
print("Aligned validation rows:", len(ensemble_df))
print("Model alignment: PASSED")

TF-IDF probabilities saved: /content/drive/MyDrive/UCU_AI_HW3/tfidf_validation_probs_v1.csv
Aligned validation rows: 55031
Model alignment: PASSED



## 3. Experiment 1 — Probability-level ensemble

I investigate whether a weighted probability ensemble of TF-IDF Logistic Regression and XLM-RoBERTa with head-tail truncation improves macro-F1.

I use a fixed stratified split of the original validation fold into calibration and evaluation subsets. I select ensemble weights on the calibration subset and report their performance on the held-out evaluation subset.

I compare the selected ensemble against both individual models on exactly the same documents. This separation reduces the risk of overstating improvements caused by hyperparameter selection.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

CLASS_ORDER = ["human", "ai", "fusion"]
SEED = 42

tfidf_probs = ensemble_df[
    ["tfidf_human", "tfidf_ai", "tfidf_fusion"]
].to_numpy(dtype=np.float64)

xlmr_probs = ensemble_df[
    ["headtail_human", "headtail_ai", "headtail_fusion"]
].to_numpy(dtype=np.float64)

y_true = ensemble_df["true_label"].to_numpy()
ids = ensemble_df["id"].astype(str).to_numpy()

assert tfidf_probs.shape == xlmr_probs.shape == (55031, 3)
assert np.allclose(tfidf_probs.sum(axis=1), 1, atol=1e-4)
assert np.allclose(xlmr_probs.sum(axis=1), 1, atol=1e-4)

val_metadata = baseline_val_df[
    ["id", "lang", "domain"]
].copy()

val_metadata["id"] = val_metadata["id"].astype(str)

aligned_metadata = ensemble_df[["id"]].merge(
    val_metadata,
    on="id",
    how="left",
    validate="one_to_one"
)

assert aligned_metadata[["lang", "domain"]].notna().all().all()

strata = (
    ensemble_df["true_label"].astype(str)
    + "_"
    + aligned_metadata["lang"].astype(str)
)

cal_idx, eval_idx = train_test_split(
    np.arange(len(ensemble_df)),
    test_size=0.5,
    random_state=SEED,
    stratify=strata
)

assert len(set(cal_idx) & set(eval_idx)) == 0

print("Calibration documents:", len(cal_idx))
print("Evaluation documents:", len(eval_idx))

for name, index in [
    ("Calibration", cal_idx),
    ("Evaluation", eval_idx)
]:
    distribution = pd.Series(y_true[index]).value_counts(
        normalize=True
    ).reindex(CLASS_ORDER)

    print(f"\n{name} class distribution:")
    display((100 * distribution).round(2))

print("Calibration/evaluation split: PASSED")

Calibration documents: 27515
Evaluation documents: 27516

Calibration class distribution:


,proportion
human,24.58
ai,44.92
fusion,30.50



Evaluation class distribution:


,proportion
human,24.59
ai,44.92
fusion,30.50


Calibration/evaluation split: PASSED


In [ ]:
from sklearn.metrics import f1_score

def predict_weighted_ensemble(weight_tfidf, indices):
    combined = (
        weight_tfidf * tfidf_probs[indices]
        + (1.0 - weight_tfidf) * xlmr_probs[indices]
    )

    return np.asarray(CLASS_ORDER)[
        np.argmax(combined, axis=1)
    ]

weight_results = []

for weight in np.linspace(0.0, 1.0, 21):

    predictions = predict_weighted_ensemble(
        weight,
        cal_idx
    )

    score = f1_score(
        y_true[cal_idx],
        predictions,
        labels=CLASS_ORDER,
        average="macro",
        zero_division=0
    )

    weight_results.append({
        "tfidf_weight": float(weight),
        "xlmr_weight": float(1.0 - weight),
        "calibration_macro_f1": float(score)
    })

weight_search_df = pd.DataFrame(weight_results)

weight_search_df = weight_search_df.sort_values(
    ["calibration_macro_f1", "tfidf_weight"],
    ascending=[False, False]
).reset_index(drop=True)

best_weight = float(
    weight_search_df.iloc[0]["tfidf_weight"]
)

print("Best TF-IDF weight:", round(best_weight, 2))
print("Best XLM-R weight:", round(1 - best_weight, 2))

print("\nTop configurations:")
display(weight_search_df.head(10).round(5))

Best TF-IDF weight: 0.6
Best XLM-R weight: 0.4

Top configurations:


,tfidf_weight,xlmr_weight,calibration_macro_f1
0,0.60,0.40,0.91611
1,0.55,0.45,0.91490
2,0.65,0.35,0.91320
3,0.50,0.50,0.91148
4,0.70,0.30,0.90829
5,0.75,0.25,0.90373
6,0.45,0.55,0.90191
7,0.80,0.20,0.89883
8,0.40,0.60,0.89536
9,0.85,0.15,0.89383


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

comparison = []

models = {
    "TF-IDF baseline": np.asarray(CLASS_ORDER)[
        np.argmax(tfidf_probs[eval_idx], axis=1)
    ],
    "XLM-R head-tail": np.asarray(CLASS_ORDER)[
        np.argmax(xlmr_probs[eval_idx], axis=1)
    ],
    "Weighted ensemble": predict_weighted_ensemble(
        best_weight,
        eval_idx
    )
}

for name, predictions in models.items():
    per_class_f1 = f1_score(
        y_true[eval_idx],
        predictions,
        labels=CLASS_ORDER,
        average=None,
        zero_division=0
    )

    comparison.append({
        "model": name,
        "macro_f1": float(np.mean(per_class_f1)),
        "accuracy": accuracy_score(y_true[eval_idx], predictions),
        "human_f1": per_class_f1[0],
        "ai_f1": per_class_f1[1],
        "fusion_f1": per_class_f1[2]
    })

evaluation_df = pd.DataFrame(comparison).sort_values(
    "macro_f1",
    ascending=False
)

print("Held-out evaluation results:")
display(evaluation_df.round(5))

print("\nSelected ensemble classification report:")
print(
    classification_report(
        y_true[eval_idx],
        models["Weighted ensemble"],
        labels=CLASS_ORDER,
        digits=4,
        zero_division=0
    )
)

best_row = evaluation_df.iloc[0]
print(
    "\nBest evaluation model:",
    best_row["model"],
    f"(macro-F1={best_row['macro_f1']:.5f})"
)

weight_search_df.to_csv(
    PROJECT_DIR / "experiment1_ensemble_weight_search.csv",
    index=False
)

evaluation_df.to_csv(
    PROJECT_DIR / "experiment1_ensemble_evaluation.csv",
    index=False
)

print("\nExperiment 1 results saved.")

Held-out evaluation results:


,model,macro_f1,accuracy,human_f1,ai_f1,fusion_f1
2,Weighted ensemble,0.91582,0.92964,0.87142,0.99299,0.88305
0,TF-IDF baseline,0.87853,0.89780,0.82006,0.98683,0.82868
1,XLM-R head-tail,0.87452,0.89323,0.81787,0.98213,0.82356



Selected ensemble classification report:
              precision    recall  f1-score   support

       human     0.8350    0.9112    0.8714      6765
          ai     0.9890    0.9970    0.9930     12359
      fusion     0.9243    0.8453    0.8831      8392

    accuracy                         0.9296     27516
   macro avg     0.9161    0.9178    0.9158     27516
weighted avg     0.9314    0.9296    0.9296     27516


Best evaluation model: Weighted ensemble (macro-F1=0.91582)

Experiment 1 results saved.



### 3.4. Group-aware ensemble robustness check

I verify the ensemble using an additional group-aware calibration and evaluation split.

The original weight search preserved label and language proportions but did not explicitly prevent related validation documents from appearing in both subsets. I therefore use the document groups established during EDA to create a stricter separation.

I repeat the ensemble weight search on the group-aware calibration subset and evaluate the selected configuration on the corresponding held-out subset. I compare the results with the original experiment to assess whether the improvement is robust.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

fold_groups = pd.read_csv(
    FOLDS_PATH,
    dtype={"id": str, "group_id": str}
)

group_metadata = ensemble_df[["id"]].merge(
    fold_groups[["id", "group_id", "cv_fold"]],
    on="id",
    how="left",
    validate="one_to_one"
)

assert group_metadata[["group_id", "cv_fold"]].notna().all().all()
assert group_metadata["cv_fold"].eq(0).all()

group_ids = group_metadata["group_id"].to_numpy()

splitter = StratifiedGroupKFold(
    n_splits=2,
    shuffle=True,
    random_state=SEED
)

group_cal_idx, group_eval_idx = next(
    splitter.split(
        X=np.zeros(len(y_true)),
        y=y_true,
        groups=group_ids
    )
)

assert set(group_ids[group_cal_idx]).isdisjoint(
    set(group_ids[group_eval_idx])
)

print("Group-aware calibration:", len(group_cal_idx))
print("Group-aware evaluation:", len(group_eval_idx))
print("Groups crossing subsets: 0")

for name, indices in [
    ("Calibration", group_cal_idx),
    ("Evaluation", group_eval_idx)
]:
    print(f"\n{name} class distribution (%):")
    display(
        (
            pd.Series(y_true[indices])
            .value_counts(normalize=True)
            .reindex(CLASS_ORDER)
            .mul(100)
            .round(2)
        )
    )

print("\nGroup-aware split: PASSED")

Group-aware calibration: 27403
Group-aware evaluation: 27628
Groups crossing subsets: 0

Calibration class distribution (%):


,proportion
human,24.74
ai,44.49
fusion,30.77



Evaluation class distribution (%):


,proportion
human,24.43
ai,45.34
fusion,30.23



Group-aware split: PASSED


In [ ]:
group_search = []

for weight in np.linspace(0, 1, 21):
    pred = predict_weighted_ensemble(
        float(weight),
        group_cal_idx
    )

    group_search.append({
        "tfidf_weight": float(weight),
        "calibration_macro_f1": f1_score(
            y_true[group_cal_idx],
            pred,
            labels=CLASS_ORDER,
            average="macro",
            zero_division=0
        )
    })

group_search_df = pd.DataFrame(group_search).sort_values(
    ["calibration_macro_f1", "tfidf_weight"],
    ascending=[False, False]
).reset_index(drop=True)

group_best_weight = float(
    group_search_df.iloc[0]["tfidf_weight"]
)

group_models = {
    "TF-IDF": np.asarray(CLASS_ORDER)[
        tfidf_probs[group_eval_idx].argmax(axis=1)
    ],
    "XLM-R head-tail": np.asarray(CLASS_ORDER)[
        xlmr_probs[group_eval_idx].argmax(axis=1)
    ],
    "Ensemble (selected)": predict_weighted_ensemble(
        group_best_weight,
        group_eval_idx
    ),
    "Ensemble (fixed 0.60)": predict_weighted_ensemble(
        0.60,
        group_eval_idx
    )
}

group_evaluation_rows = []

for name, predictions in group_models.items():
    class_f1 = f1_score(
        y_true[group_eval_idx],
        predictions,
        labels=CLASS_ORDER,
        average=None,
        zero_division=0
    )

    group_evaluation_rows.append({
        "model": name,
        "macro_f1": float(class_f1.mean()),
        "human_f1": float(class_f1[0]),
        "ai_f1": float(class_f1[1]),
        "fusion_f1": float(class_f1[2])
    })

group_evaluation_df = pd.DataFrame(
    group_evaluation_rows
).sort_values("macro_f1", ascending=False)

print("Best group-aware TF-IDF weight:", group_best_weight)
print("\nGroup-aware evaluation results:")
display(group_evaluation_df.round(5))

group_search_df.to_csv(
    PROJECT_DIR / "experiment1_group_weight_search.csv",
    index=False
)

group_evaluation_df.to_csv(
    PROJECT_DIR / "experiment1_group_evaluation.csv",
    index=False
)

Best group-aware TF-IDF weight: 0.6000000000000001

Group-aware evaluation results:


,model,macro_f1,human_f1,ai_f1,fusion_f1
3,Ensemble (fixed 0.60),0.91870,0.87599,0.99320,0.88689
2,Ensemble (selected),0.91870,0.87599,0.99320,0.88689
0,TF-IDF,0.87766,0.81928,0.98634,0.82735
1,XLM-R head-tail,0.87639,0.82101,0.98238,0.82579



## 5. Experiment 1 — Kaggle submission

I generate a Kaggle submission using the selected probability-level ensemble: 60% word-level TF-IDF Logistic Regression and 40% XLM-RoBERTa with head-tail truncation.

I use the XLM-RoBERTa checkpoint associated with the validated head-tail experiment and preserve the exact class probability order: human, ai, fusion.

I generate test probabilities independently for both models, align predictions by document ID, and combine them using the selected ensemble weights.

I validate the final CSV against the official sample submission before submitting it to Kaggle.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_DIR = Path("/content/drive/MyDrive/UCU_AI_HW3")

TEST_PATH = PROJECT_DIR / "test.csv"
ENSEMBLE_WEIGHT = 0.60
CLASS_ORDER = ["human", "ai", "fusion"]

assert TEST_PATH.exists(), "test.csv not found in Google Drive."
assert "baseline_model" in globals(), (
    "The fold-trained TF-IDF model is not in memory. "
    "Run Step 2.2 before continuing."
)

test_df = pd.read_csv(
    TEST_PATH,
    dtype={"id": str}
).reset_index(drop=True)

assert len(test_df) == 25_000
assert test_df["id"].is_unique
assert test_df["text"].notna().all()

print("Test documents:", len(test_df))
print("Ensemble weights:")
print("TF-IDF:", ENSEMBLE_WEIGHT)
print("XLM-R:", 1 - ENSEMBLE_WEIGHT)
print("Class order:", CLASS_ORDER)

Test documents: 25000
Ensemble weights:
TF-IDF: 0.6
XLM-R: 0.4
Class order: ['human', 'ai', 'fusion']


In [ ]:
TFIDF_TEST_PATH = (
    PROJECT_DIR / "experiment1_tfidf_test_probs.csv"
)

if TFIDF_TEST_PATH.exists():
    tfidf_test_df = pd.read_csv(
        TFIDF_TEST_PATH,
        dtype={"id": str}
    )
    print("Loaded existing TF-IDF test probabilities.")
else:
    tfidf_test_probs = baseline_model.predict_proba(
        test_df["text"]
    )

    model_classes = list(
        baseline_model.named_steps["classifier"].classes_
    )

    class_indices = [
        model_classes.index(label)
        for label in CLASS_ORDER
    ]

    tfidf_test_probs = tfidf_test_probs[:, class_indices]

    tfidf_test_df = pd.DataFrame({
        "id": test_df["id"],
        "tfidf_human": tfidf_test_probs[:, 0],
        "tfidf_ai": tfidf_test_probs[:, 1],
        "tfidf_fusion": tfidf_test_probs[:, 2]
    })

    tfidf_test_df.to_csv(TFIDF_TEST_PATH, index=False)

assert len(tfidf_test_df) == 25_000
assert tfidf_test_df["id"].equals(test_df["id"])
assert tfidf_test_df["id"].is_unique

prob_columns = ["tfidf_human", "tfidf_ai", "tfidf_fusion"]

assert np.isfinite(tfidf_test_df[prob_columns].to_numpy()).all()
assert np.allclose(
    tfidf_test_df[prob_columns].sum(axis=1),
    1.0,
    atol=1e-4
)

print("TF-IDF probabilities saved:", TFIDF_TEST_PATH)
print("Documents:", len(tfidf_test_df))
print("Probability verification: PASSED")

TF-IDF probabilities saved: /content/drive/MyDrive/UCU_AI_HW3/experiment1_tfidf_test_probs.csv
Documents: 25000
Probability verification: PASSED



### 5.3. Restore XLM-RoBERTa for test inference

I restore the XLM-RoBERTa checkpoint used in the validated head-tail experiment.

I use the same tokenizer and input construction as in the encoder notebook: the first 255 and last 255 content tokens, surrounded by the model's special tokens.

I keep the model in evaluation mode and generate class probabilities without updating its parameters. All predictions will be saved separately from previous XLM-RoBERTa runs.


In [ ]:
import torch
import gc

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LENGTH = 512

CHECKPOINT_DIR = (
    PROJECT_DIR / "xlmr_base_40k_512" / "checkpoint-2500"
)

assert CHECKPOINT_DIR.is_dir(), "Checkpoint-2500 not found."
assert (
    (CHECKPOINT_DIR / "model.safetensors").exists()
    or (CHECKPOINT_DIR / "pytorch_model.bin").exists()
), "Checkpoint weights are missing."

assert torch.cuda.is_available(), (
    "Enable a GPU runtime before running XLM-R inference."
)

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

xlmr_model = AutoModelForSequenceClassification.from_pretrained(
    str(CHECKPOINT_DIR)
)

xlmr_model.to("cuda")
xlmr_model.eval()

id2label = {
    int(k): v
    for k, v in xlmr_model.config.id2label.items()
}

assert [id2label[i] for i in range(3)] == CLASS_ORDER

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

print("Checkpoint:", CHECKPOINT_DIR.name)
print("GPU:", torch.cuda.get_device_name(0))
print("Class mapping:", id2label)
print("XLM-R model ready for head-tail inference.")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Checkpoint: checkpoint-2500
GPU: Tesla T4
Class mapping: {0: 'human', 1: 'ai', 2: 'fusion'}
XLM-R model ready for head-tail inference.



### 5.4. Resumable head-tail inference

I generate XLM-RoBERTa class probabilities for the complete competition test set using head-tail tokenization.

I process the documents in chunks of 500, use mixed-precision GPU inference, and save the probability outputs after each chunk. Saved chunks are checked for correct document IDs, probability values, and class order before reuse.

This ensures that interrupted inference can resume without repeating completed work.


In [ ]:
import os
import json
import numpy as np
import pandas as pd

from datasets import Dataset
from scipy.special import softmax
from tqdm.auto import tqdm
from transformers import Trainer, TrainingArguments

CHUNK_SIZE = 500

XLMR_PROBS_DIR = (
    PROJECT_DIR / "experiment1_xlmr_checkpoint2500_headtail_probs"
)
XLMR_PROBS_DIR.mkdir(parents=True, exist_ok=True)

def tokenize_head_tail_test(batch):
    encoded = tokenizer(
        batch["text"],
        add_special_tokens=False,
        truncation=False,
        padding=False,
        verbose=False
    )

    input_ids_batch = []

    for token_ids in encoded["input_ids"]:
        if len(token_ids) > 510:
            token_ids = token_ids[:255] + token_ids[-255:]

        final_ids = (
            [tokenizer.cls_token_id]
            + token_ids
            + [tokenizer.sep_token_id]
        )

        assert len(final_ids) <= 512
        input_ids_batch.append(final_ids)

    return {
        "input_ids": input_ids_batch,
        "attention_mask": [
            [1] * len(ids) for ids in input_ids_batch
        ]
    }

inference_args = TrainingArguments(
    output_dir="/content/experiment1_xlmr_inference",
    per_device_eval_batch_size=8,
    fp16=True,
    report_to="none",
    dataloader_num_workers=2
)

xlmr_inference_trainer = Trainer(
    model=xlmr_model,
    args=inference_args,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint:", CHECKPOINT_DIR.name)
print("Head-tail tokens: 255 + 255")
print("Batch size: 8")
print("Chunks:", (len(test_df) + CHUNK_SIZE - 1) // CHUNK_SIZE)
print("Output:", XLMR_PROBS_DIR)

GPU: Tesla T4
Checkpoint: checkpoint-2500
Head-tail tokens: 255 + 255
Batch size: 8
Chunks: 50
Output: /content/drive/MyDrive/UCU_AI_HW3/experiment1_xlmr_checkpoint2500_headtail_probs


In [ ]:
PROB_COLUMNS = [
    "xlmr_human",
    "xlmr_ai",
    "xlmr_fusion"
]

num_chunks = (len(test_df) + CHUNK_SIZE - 1) // CHUNK_SIZE

for chunk_idx in tqdm(range(num_chunks), desc="XLM-R head-tail"):

    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(test_df))

    chunk_path = (
        XLMR_PROBS_DIR / f"probs_{chunk_idx:03d}.csv"
    )

    chunk_df = test_df.iloc[start:end]
    expected_ids = chunk_df["id"].astype(str).reset_index(
        drop=True
    )

    if chunk_path.exists():
        try:
            saved = pd.read_csv(chunk_path, dtype={"id": str})
            saved_probs = saved[PROB_COLUMNS].to_numpy()

            valid = (
                list(saved.columns) == ["id"] + PROB_COLUMNS
                and saved["id"].equals(expected_ids)
                and np.isfinite(saved_probs).all()
                and (saved_probs >= 0).all()
                and np.allclose(
                    saved_probs.sum(axis=1), 1, atol=1e-4
                )
            )

            if valid:
                continue
        except Exception:
            pass

    chunk_dataset = Dataset.from_pandas(
        chunk_df[["text"]],
        preserve_index=False
    )

    chunk_tokenized = chunk_dataset.map(
        tokenize_head_tail_test,
        batched=True,
        batch_size=64,
        remove_columns=["text"],
        desc=f"Tokenizing chunk {chunk_idx + 1}"
    )

    output = xlmr_inference_trainer.predict(chunk_tokenized)

    logits = output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    probabilities = softmax(
        np.asarray(logits, dtype=np.float64),
        axis=1
    )

    assert probabilities.shape == (end - start, 3)

    chunk_probs = pd.DataFrame({
        "id": expected_ids,
        "xlmr_human": probabilities[:, 0],
        "xlmr_ai": probabilities[:, 1],
        "xlmr_fusion": probabilities[:, 2]
    })

    temporary_path = chunk_path.with_suffix(".tmp")
    chunk_probs.to_csv(temporary_path, index=False)
    os.replace(temporary_path, chunk_path)

print("\nCompleted probability chunks:",
      len(list(XLMR_PROBS_DIR.glob("probs_*.csv"))),
      "/", num_chunks)

XLM-R head-tail:   0%|          | 0/50 [00:00<?, ?it/s]

Tokenizing chunk 1:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing chunk 2:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing chunk 3:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing chunk 4:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing chunk 5:   0%|          | 0/500 [00:00<?, ? examples/s]

... (42 outputs left out)

Tokenizing chunk 48:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing chunk 49:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing chunk 50:   0%|          | 0/500 [00:00<?, ? examples/s]


Completed probability chunks: 50 / 50



### 5.5. Assemble and validate the ensemble submission

I combine the test probabilities from TF-IDF Logistic Regression and XLM-RoBERTa using the ensemble weights selected on validation data.

I use 60% TF-IDF and 40% XLM-RoBERTa with head-tail truncation. I align probability vectors by document ID and verify that all 25,000 test examples are represented exactly once.

I preserve the official sample-submission order and save the ensemble probabilities separately for subsequent experiments.


In [ ]:
import numpy as np
import pandas as pd

CLASS_ORDER = ["human", "ai", "fusion"]
ENSEMBLE_WEIGHT = 0.60

chunk_paths = [
    XLMR_PROBS_DIR / f"probs_{i:03d}.csv"
    for i in range(50)
]

assert all(p.exists() for p in chunk_paths), (
    "Some XLM-R probability chunks are missing."
)

xlmr_test_df = pd.concat(
    [
        pd.read_csv(p, dtype={"id": str})
        for p in chunk_paths
    ],
    ignore_index=True
)

tfidf_test_df = pd.read_csv(
    PROJECT_DIR / "experiment1_tfidf_test_probs.csv",
    dtype={"id": str}
)

assert len(xlmr_test_df) == 25_000
assert len(tfidf_test_df) == 25_000
assert xlmr_test_df["id"].is_unique
assert tfidf_test_df["id"].is_unique

combined_df = tfidf_test_df.merge(
    xlmr_test_df,
    on="id",
    how="inner",
    validate="one_to_one"
)

assert len(combined_df) == 25_000

tfidf_test_probs = combined_df[
    ["tfidf_human", "tfidf_ai", "tfidf_fusion"]
].to_numpy(dtype=np.float64)

xlmr_test_probs = combined_df[
    ["xlmr_human", "xlmr_ai", "xlmr_fusion"]
].to_numpy(dtype=np.float64)

for probs in [tfidf_test_probs, xlmr_test_probs]:
    assert np.isfinite(probs).all()
    assert (probs >= 0).all()
    assert np.allclose(probs.sum(axis=1), 1, atol=1e-4)

ensemble_test_probs = (
    ENSEMBLE_WEIGHT * tfidf_test_probs
    + (1.0 - ENSEMBLE_WEIGHT) * xlmr_test_probs
)

assert np.allclose(
    ensemble_test_probs.sum(axis=1),
    1.0,
    atol=1e-4
)

combined_df["ensemble_human"] = ensemble_test_probs[:, 0]
combined_df["ensemble_ai"] = ensemble_test_probs[:, 1]
combined_df["ensemble_fusion"] = ensemble_test_probs[:, 2]

probabilities_path = (
    PROJECT_DIR / "experiment1_ensemble_test_probs.csv"
)

combined_df.to_csv(probabilities_path, index=False)

sample_candidates = [
    PROJECT_DIR / "sample_submission.csv",
    DATA_DIR / "sample_submission.csv"
]

sample_path = next(
    (p for p in sample_candidates if p.exists()),
    None
)

assert sample_path is not None, (
    "Official sample_submission.csv is missing. "
    "Extract it from the Kaggle competition archive."
)

sample_submission = pd.read_csv(
    sample_path,
    dtype={"id": str}
)

assert len(sample_submission) == 25_000
assert sample_submission["id"].is_unique

submission_probs = sample_submission[["id"]].merge(
    combined_df[
        ["id", "ensemble_human", "ensemble_ai", "ensemble_fusion"]
    ],
    on="id",
    how="left",
    validate="one_to_one",
    sort=False
)

assert submission_probs[
    ["ensemble_human", "ensemble_ai", "ensemble_fusion"]
].notna().all().all()

predicted_indices = np.argmax(
    submission_probs[
        ["ensemble_human", "ensemble_ai", "ensemble_fusion"]
    ].to_numpy(),
    axis=1
)

submission = pd.DataFrame({
    "id": submission_probs["id"],
    "label": np.asarray(CLASS_ORDER)[predicted_indices]
})

assert submission.shape == (25_000, 2)
assert list(submission.columns) == ["id", "label"]
assert submission["id"].equals(sample_submission["id"])
assert submission["id"].is_unique
assert submission["label"].isin(CLASS_ORDER).all()

submission_path = (
    PROJECT_DIR /
    "submission_experiment1_ensemble_60tfidf_40xlmr_headtail.csv"
)

submission.to_csv(submission_path, index=False)

print("Submission verification: PASSED")
print("Submission path:", submission_path)
print("Test probabilities saved:", probabilities_path)
print("Number of predictions:", len(submission))

print("\nPrediction distribution:")
display(submission["label"].value_counts())

display(submission.head())

Submission verification: PASSED
Submission path: /content/drive/MyDrive/UCU_AI_HW3/submission_experiment1_ensemble_60tfidf_40xlmr_headtail.csv
Test probabilities saved: /content/drive/MyDrive/UCU_AI_HW3/experiment1_ensemble_test_probs.csv
Number of predictions: 25000

Prediction distribution:


,count
label,
ai,12972
human,6695
fusion,5333


,id,label
0,8ff2d1,ai
1,a261cf,human
2,ad5614,fusion
3,3a23ac,ai
4,28b4e5,ai



### 5.6. Submit the ensemble to Kaggle

I submit the validated 60/40 probability ensemble to Kaggle and compare its public macro-F1 with the original TF-IDF baseline.

I use the public leaderboard as an additional diagnostic while keeping validation-based model selection separate from leaderboard feedback.


In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = (
    userdata.get("kaggle_token").strip()
)

COMPETITION = "ai-generated-text-detection-hw3-ucu-ai-2026"

assert submission_path.exists()

!kaggle competitions submit \
    -c {COMPETITION} \
    -f {submission_path} \
    -m "Experiment 1: TF-IDF 60% + XLM-R head-tail 40%, checkpoint 2500"

100% 285k/285k [00:01<00:00, 209kB/s]
Successfully submitted to AI-Generated Text Detection | HW3 | UCU AI 2026


## 6. Experiment 2 — Class-specific decision calibration

I investigate whether class-specific decision biases can improve the macro-F1 of the selected probability ensemble.

I keep the ensemble fixed at 60% TF-IDF Logistic Regression and 40% XLM-RoBERTa with head-tail truncation. Instead of using the default argmax decision rule, I adjust log-probability scores for the `human` and `fusion` classes, keeping `ai` as the reference.

I select calibration parameters on the group-aware calibration subset and evaluate them on the corresponding held-out evaluation subset.

I compare macro-F1 and per-class F1 against the uncalibrated ensemble and retain calibration only if the improvement generalizes. Since the ensemble itself was previously studied on this validation split, these results are treated as additional internal evidence rather than an unbiased final test estimate.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)
from IPython.display import display

CLASS_ORDER = ["human", "ai", "fusion"]
ENSEMBLE_WEIGHT = 0.60

ensemble_probs = (
    ENSEMBLE_WEIGHT * tfidf_probs
    + (1.0 - ENSEMBLE_WEIGHT) * xlmr_probs
)

assert ensemble_probs.shape == (55_031, 3)
assert np.isfinite(ensemble_probs).all()
assert np.allclose(
    ensemble_probs.sum(axis=1),
    1.0,
    atol=1e-4
)

log_probs = np.log(
    np.clip(ensemble_probs, 1e-12, 1.0)
)

def predict_with_bias(
    indices,
    human_bias=0.0,
    fusion_bias=0.0
):
    scores = log_probs[indices].copy()

    scores[:, 0] += human_bias
    scores[:, 2] += fusion_bias

    predicted_indices = np.argmax(scores, axis=1)

    return np.asarray(CLASS_ORDER)[predicted_indices]

reference_predictions = predict_with_bias(
    group_eval_idx
)

reference_f1 = f1_score(
    y_true[group_eval_idx],
    reference_predictions,
    labels=CLASS_ORDER,
    average="macro",
    zero_division=0
)

print(f"Reference ensemble macro-F1: {reference_f1:.5f}")
print("Calibration documents:", len(group_cal_idx))
print("Evaluation documents:", len(group_eval_idx))

assert abs(reference_f1 - 0.91870) < 0.0001

print("Calibration setup: PASSED")

Reference ensemble macro-F1: 0.91870
Calibration documents: 27403
Evaluation documents: 27628
Calibration setup: PASSED


In [ ]:
from itertools import product

bias_grid = np.round(
    np.linspace(-0.20, 0.20, 9),
    3
)

calibration_results = []

for human_bias, fusion_bias in product(
    bias_grid,
    bias_grid
):
    predictions = predict_with_bias(
        group_cal_idx,
        human_bias=float(human_bias),
        fusion_bias=float(fusion_bias)
    )

    macro_f1 = f1_score(
        y_true[group_cal_idx],
        predictions,
        labels=CLASS_ORDER,
        average="macro",
        zero_division=0
    )

    calibration_results.append({
        "human_bias": float(human_bias),
        "fusion_bias": float(fusion_bias),
        "calibration_macro_f1": float(macro_f1),
        "adjustment_size": float(
            abs(human_bias) + abs(fusion_bias)
        )
    })

calibration_search_df = pd.DataFrame(
    calibration_results
).sort_values(
    ["calibration_macro_f1", "adjustment_size"],
    ascending=[False, True]
).reset_index(drop=True)

best_human_bias = float(
    calibration_search_df.iloc[0]["human_bias"]
)

best_fusion_bias = float(
    calibration_search_df.iloc[0]["fusion_bias"]
)

reference_calibration_f1 = f1_score(
    y_true[group_cal_idx],
    predict_with_bias(group_cal_idx),
    labels=CLASS_ORDER,
    average="macro",
    zero_division=0
)

print("Reference calibration macro-F1:",
      round(reference_calibration_f1, 5))
print("Best human bias:", best_human_bias)
print("Best fusion bias:", best_fusion_bias)

print("\nTop calibration configurations:")
display(
    calibration_search_df[
        ["human_bias", "fusion_bias", "calibration_macro_f1"]
    ].head(10).round(5)
)

calibration_search_df.to_csv(
    PROJECT_DIR / "experiment2_bias_search.csv",
    index=False
)

Reference calibration macro-F1: 0.91325
Best human bias: 0.2
Best fusion bias: 0.2

Top calibration configurations:


,human_bias,fusion_bias,calibration_macro_f1
0,0.20,0.20,0.91393
1,0.15,0.15,0.91377
2,0.20,0.15,0.91357
3,0.10,0.10,0.91343
4,0.20,0.05,0.91342
5,0.05,0.05,0.91341
6,0.00,0.00,0.91325
7,0.20,0.10,0.91320
8,0.15,0.10,0.91318
9,0.15,0.00,0.91314


In [ ]:
evaluation_predictions = {
    "Uncalibrated ensemble": predict_with_bias(
        group_eval_idx
    ),
    "Calibrated ensemble": predict_with_bias(
        group_eval_idx,
        human_bias=best_human_bias,
        fusion_bias=best_fusion_bias
    )
}

evaluation_results = []

for model_name, predictions in evaluation_predictions.items():

    per_class_f1 = f1_score(
        y_true[group_eval_idx],
        predictions,
        labels=CLASS_ORDER,
        average=None,
        zero_division=0
    )

    evaluation_results.append({
        "model": model_name,
        "macro_f1": float(np.mean(per_class_f1)),
        "accuracy": float(
            accuracy_score(y_true[group_eval_idx], predictions)
        ),
        "human_f1": float(per_class_f1[0]),
        "ai_f1": float(per_class_f1[1]),
        "fusion_f1": float(per_class_f1[2])
    })

calibration_evaluation_df = pd.DataFrame(
    evaluation_results
)

print("Held-out evaluation results:")
display(calibration_evaluation_df.round(5))

improvement = (
    calibration_evaluation_df.iloc[1]["macro_f1"]
    - calibration_evaluation_df.iloc[0]["macro_f1"]
)

print(f"\nMacro-F1 improvement: {improvement:+.5f}")

print("\nCalibrated ensemble classification report:")
print(
    classification_report(
        y_true[group_eval_idx],
        evaluation_predictions["Calibrated ensemble"],
        labels=CLASS_ORDER,
        digits=4,
        zero_division=0
    )
)

calibration_evaluation_df.to_csv(
    PROJECT_DIR / "experiment2_bias_evaluation.csv",
    index=False
)

print("\nExperiment 2 evaluation saved.")

Held-out evaluation results:


,model,macro_f1,accuracy,human_f1,ai_f1,fusion_f1
0,Uncalibrated ensemble,0.91870,0.93253,0.87599,0.99320,0.88689
1,Calibrated ensemble,0.91917,0.93297,0.87604,0.99383,0.88763



Macro-F1 improvement: +0.00047

Calibrated ensemble classification report:
              precision    recall  f1-score   support

       human     0.8407    0.9145    0.8760      6750
          ai     0.9909    0.9968    0.9938     12526
      fusion     0.9262    0.8521    0.8876      8352

    accuracy                         0.9330     27628
   macro avg     0.9192    0.9212    0.9192     27628
weighted avg     0.9346    0.9330    0.9329     27628


Experiment 2 evaluation saved.



## 7. Experiment 3 — Length-aware probability ensemble

I investigate whether the optimal combination of TF-IDF Logistic Regression and XLM-RoBERTa depends on document length.

The classical model uses lexical information from the complete document, whereas XLM-RoBERTa uses head-tail truncation with a maximum of 512 tokens. Their relative strengths may therefore differ across short and long texts.

I divide documents into three groups based on word count and independently select the TF-IDF ensemble weight for each group using the group-aware calibration subset.

I then compare the length-aware ensemble against the fixed 60/40 ensemble on the held-out evaluation subset, without modifying the trained models or using test labels.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from IPython.display import display

CLASS_ORDER = ["human", "ai", "fusion"]

validation_texts = ensemble_df[["id"]].merge(
    baseline_val_df[["id", "text"]],
    on="id",
    how="left",
    validate="one_to_one",
    sort=False
)

assert len(validation_texts) == len(ensemble_df)
assert validation_texts["text"].notna().all()

word_counts = (
    validation_texts["text"]
    .str.count(r"\S+")
    .to_numpy()
)

def get_length_groups(counts):
    counts = np.asarray(counts)

    return np.select(
        [
            counts <= 256,
            (counts > 256) & (counts <= 1024)
        ],
        [
            "short",
            "medium"
        ],
        default="long"
    )

length_groups = get_length_groups(word_counts)

length_summary = pd.DataFrame({
    "length_group": length_groups,
    "word_count": word_counts,
    "label": y_true
})

print("Validation document-length groups:")

display(
    length_summary.groupby("length_group")
    .agg(
        documents=("word_count", "size"),
        median_words=("word_count", "median")
    )
    .reindex(["short", "medium", "long"])
)

for group in ["short", "medium", "long"]:
    cal_count = int(
        np.sum(length_groups[group_cal_idx] == group)
    )
    eval_count = int(
        np.sum(length_groups[group_eval_idx] == group)
    )

    print(
        f"{group}: calibration={cal_count}, "
        f"evaluation={eval_count}"
    )

    assert cal_count >= 100
    assert eval_count >= 100

print("\nLength-aware experiment setup: PASSED")

Validation document-length groups:


,documents,median_words
length_group,,
short,12587,145.0
medium,37570,517.0
long,4874,1351.0


short: calibration=6287, evaluation=6300
medium: calibration=18711, evaluation=18859
long: calibration=2405, evaluation=2469

Length-aware experiment setup: PASSED


In [ ]:
WEIGHT_GRID = np.round(np.linspace(0, 1, 21), 2)

length_weight_results = []

for group in ["short", "medium", "long"]:

    indices = group_cal_idx[
        length_groups[group_cal_idx] == group
    ]

    for weight in WEIGHT_GRID:

        predictions = predict_weighted_ensemble(
            float(weight),
            indices
        )

        score = f1_score(
            y_true[indices],
            predictions,
            labels=CLASS_ORDER,
            average="macro",
            zero_division=0
        )

        length_weight_results.append({
            "length_group": group,
            "tfidf_weight": float(weight),
            "xlmr_weight": float(1 - weight),
            "calibration_macro_f1": float(score),
            "distance_from_reference": abs(
                float(weight) - 0.60
            )
        })

length_weight_search_df = pd.DataFrame(
    length_weight_results
)

best_length_weights_df = (
    length_weight_search_df
    .sort_values(
        ["calibration_macro_f1", "distance_from_reference"],
        ascending=[False, True]
    )
    .drop_duplicates("length_group")
    .set_index("length_group")
    .loc[["short", "medium", "long"]]
    .reset_index()
)

best_length_weights = dict(
    zip(
        best_length_weights_df["length_group"],
        best_length_weights_df["tfidf_weight"]
    )
)

print("Selected length-dependent weights:")
display(
    best_length_weights_df[
        [
            "length_group",
            "tfidf_weight",
            "xlmr_weight",
            "calibration_macro_f1"
        ]
    ].round(5)
)

length_weight_search_df.to_csv(
    PROJECT_DIR / "experiment3_length_weight_search.csv",
    index=False
)

print("\nLength-specific weight search: PASSED")

Selected length-dependent weights:


,length_group,tfidf_weight,xlmr_weight,calibration_macro_f1
0,short,0.50,0.50,0.86753
1,medium,0.60,0.40,0.93225
2,long,0.65,0.35,0.91174



Length-specific weight search: PASSED



### 7.3. Held-out evaluation of length-aware ensembling

I evaluate the selected length-dependent weights on the group-aware evaluation subset, which was not used to choose the weights.

I compare the length-aware ensemble against the fixed 60/40 ensemble and the individual models. I report overall macro-F1, per-class F1, and performance within each document-length group.

I use the same evaluation documents for every configuration and retain the length-aware strategy only if its improvement generalizes beyond the calibration subset.


In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import pandas as pd
from IPython.display import display

def predict_length_aware(indices):
    indices = np.asarray(indices)

    weights = np.array([
        best_length_weights[group]
        for group in length_groups[indices]
    ], dtype=np.float64)

    combined_probs = (
        weights[:, None] * tfidf_probs[indices]
        + (1.0 - weights[:, None]) * xlmr_probs[indices]
    )

    return np.asarray(CLASS_ORDER)[
        np.argmax(combined_probs, axis=1)
    ]

eval_predictions = {
    "TF-IDF": np.asarray(CLASS_ORDER)[
        np.argmax(tfidf_probs[group_eval_idx], axis=1)
    ],
    "XLM-R head-tail": np.asarray(CLASS_ORDER)[
        np.argmax(xlmr_probs[group_eval_idx], axis=1)
    ],
    "Fixed 60/40 ensemble": predict_weighted_ensemble(
        0.60, group_eval_idx
    ),
    "Length-aware ensemble": predict_length_aware(
        group_eval_idx
    )
}

length_eval_rows = []

for name, predictions in eval_predictions.items():
    per_class_f1 = f1_score(
        y_true[group_eval_idx],
        predictions,
        labels=CLASS_ORDER,
        average=None,
        zero_division=0
    )

    length_eval_rows.append({
        "model": name,
        "macro_f1": float(per_class_f1.mean()),
        "accuracy": float(accuracy_score(
            y_true[group_eval_idx], predictions
        )),
        "human_f1": float(per_class_f1[0]),
        "ai_f1": float(per_class_f1[1]),
        "fusion_f1": float(per_class_f1[2])
    })

length_evaluation_df = pd.DataFrame(
    length_eval_rows
).sort_values(
    "macro_f1", ascending=False
).reset_index(drop=True)

print("Group-aware held-out evaluation:")
display(length_evaluation_df.round(5))

reference_f1 = length_evaluation_df.loc[
    length_evaluation_df["model"] == "Fixed 60/40 ensemble",
    "macro_f1"
].iloc[0]

length_aware_f1 = length_evaluation_df.loc[
    length_evaluation_df["model"] == "Length-aware ensemble",
    "macro_f1"
].iloc[0]

print(
    f"\nImprovement over fixed 60/40: "
    f"{length_aware_f1 - reference_f1:+.5f}"
)

length_evaluation_df.to_csv(
    PROJECT_DIR / "experiment3_length_evaluation.csv",
    index=False
)

Group-aware held-out evaluation:


,model,macro_f1,accuracy,human_f1,ai_f1,fusion_f1
0,Length-aware ensemble,0.91935,0.93300,0.87725,0.99289,0.88790
1,Fixed 60/40 ensemble,0.91870,0.93253,0.87599,0.99320,0.88689
2,TF-IDF,0.87766,0.89775,0.81928,0.98634,0.82735
3,XLM-R head-tail,0.87639,0.89550,0.82101,0.98238,0.82579



Improvement over fixed 60/40: +0.00065


In [ ]:
length_group_results = []

for group in ["short", "medium", "long"]:
    mask = length_groups[group_eval_idx] == group

    indices = group_eval_idx[mask]

    fixed_predictions = predict_weighted_ensemble(
        0.60, indices
    )

    adaptive_predictions = predict_length_aware(
        indices
    )

    fixed_f1 = f1_score(
        y_true[indices],
        fixed_predictions,
        labels=CLASS_ORDER,
        average="macro",
        zero_division=0
    )

    adaptive_f1 = f1_score(
        y_true[indices],
        adaptive_predictions,
        labels=CLASS_ORDER,
        average="macro",
        zero_division=0
    )

    length_group_results.append({
        "length_group": group,
        "documents": len(indices),
        "selected_tfidf_weight": best_length_weights[group],
        "fixed_macro_f1": fixed_f1,
        "length_aware_macro_f1": adaptive_f1,
        "improvement": adaptive_f1 - fixed_f1
    })

length_group_evaluation_df = pd.DataFrame(
    length_group_results
)

print("Held-out results by document length:")
display(length_group_evaluation_df.round(5))

length_group_evaluation_df.to_csv(
    PROJECT_DIR / "experiment3_length_group_evaluation.csv",
    index=False
)

print("\nExperiment 3 evaluation completed.")

Held-out results by document length:


,length_group,documents,selected_tfidf_weight,fixed_macro_f1,length_aware_macro_f1,improvement
0,short,6300,0.50,0.88038,0.88170,0.00132
1,medium,18859,0.60,0.93476,0.93476,0.00000
2,long,2469,0.65,0.90791,0.91060,0.00269



Experiment 3 evaluation completed.



## 8. Experiment 4 — Stylometry-enhanced text classification

I investigate whether document-level stylometric features provide complementary information to word-level TF-IDF representations.

I extract numerical features describing text length, punctuation usage, sentence structure, and lexical diversity. I fit all preprocessing transformations exclusively on the training folds.

I compare a stylometry-enhanced Logistic Regression classifier against the established TF-IDF baseline on the same fixed validation split.

If the new model provides useful complementary predictions, I will evaluate its contribution to the existing probability ensemble.


In [ ]:
import re
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

tqdm.pandas()

FEATURE_NAMES = [
    "log_char_count",
    "log_word_count",
    "mean_word_length",
    "lexical_diversity",
    "punctuation_ratio",
    "digit_ratio",
    "uppercase_ratio",
    "newline_ratio",
    "comma_per_word",
    "period_per_word",
    "exclamation_per_word",
    "question_per_word",
    "mean_sentence_words",
    "sentence_length_std",
]

def extract_stylometric_features(text):
    text = str(text)

    words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    word_count = len(words)
    char_count = len(text)

    sentences = [
        s for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

    sentence_lengths = [
        len(re.findall(r"\b\w+\b", s))
        for s in sentences
    ]

    lexical_diversity = (
        len(set(word.lower() for word in words))
        / max(word_count, 1)
    )

    return [
        np.log1p(char_count),
        np.log1p(word_count),
        sum(map(len, words)) / max(word_count, 1),
        lexical_diversity,
        sum(not c.isalnum() and not c.isspace()
            for c in text) / max(char_count, 1),
        sum(c.isdigit() for c in text) / max(char_count, 1),
        sum(c.isupper() for c in text) / max(char_count, 1),
        text.count("\n") / max(char_count, 1),
        text.count(",") / max(word_count, 1),
        text.count(".") / max(word_count, 1),
        text.count("!") / max(word_count, 1),
        text.count("?") / max(word_count, 1),
        float(np.mean(sentence_lengths))
            if sentence_lengths else 0.0,
        float(np.std(sentence_lengths))
            if sentence_lengths else 0.0,
    ]

print("Number of stylometric features:", len(FEATURE_NAMES))

sample_features = extract_stylometric_features(
    "This is a test sentence. Here is another one!"
)

assert len(sample_features) == len(FEATURE_NAMES)
assert np.isfinite(sample_features).all()

print("Feature extraction verification: PASSED")

Number of stylometric features: 14
Feature extraction verification: PASSED



### 8.2. Stylometric feature extraction and caching

I extract 14 document-level stylometric features from the original training and validation texts.

The features describe document length, lexical diversity, punctuation, character composition, and sentence structure.

I cache the resulting feature matrices in Google Drive to make subsequent experiments reproducible and avoid repeated preprocessing.

I preserve document identifiers and verify feature alignment before combining the stylometric features with TF-IDF representations.


In [ ]:
import numpy as np
import pandas as pd
import time

from pathlib import Path
from tqdm.auto import tqdm

STYLOMETRY_PATH = (
    PROJECT_DIR / "experiment4_stylometric_features.npz"
)

assert len(baseline_train_df) == 217_321
assert len(baseline_val_df) == 55_031

train_ids = baseline_train_df["id"].astype(str).to_numpy()
val_ids = baseline_val_df["id"].astype(str).to_numpy()

if STYLOMETRY_PATH.exists():

    cached = np.load(STYLOMETRY_PATH, allow_pickle=False)

    assert np.array_equal(
        cached["train_ids"].astype(str),
        train_ids
    )
    assert np.array_equal(
        cached["val_ids"].astype(str),
        val_ids
    )
    assert list(cached["feature_names"]) == FEATURE_NAMES

    X_train_style = cached["X_train_style"]
    X_val_style = cached["X_val_style"]

    cached.close()

    print("Loaded cached stylometric features.")

else:

    start_time = time.perf_counter()

    def extract_feature_matrix(texts, description):
        features = [
            extract_stylometric_features(text)
            for text in tqdm(texts, desc=description)
        ]

        return np.asarray(features, dtype=np.float32)

    X_train_style = extract_feature_matrix(
        baseline_train_df["text"],
        "Training stylometry"
    )

    X_val_style = extract_feature_matrix(
        baseline_val_df["text"],
        "Validation stylometry"
    )

    np.savez_compressed(
        STYLOMETRY_PATH,
        train_ids=train_ids,
        val_ids=val_ids,
        feature_names=np.asarray(FEATURE_NAMES),
        X_train_style=X_train_style,
        X_val_style=X_val_style
    )

    print(
        "Extraction time:",
        round((time.perf_counter() - start_time) / 60, 2),
        "minutes"
    )

assert X_train_style.shape == (217_321, len(FEATURE_NAMES))
assert X_val_style.shape == (55_031, len(FEATURE_NAMES))

assert np.isfinite(X_train_style).all()
assert np.isfinite(X_val_style).all()

print("\nTraining features:", X_train_style.shape)
print("Validation features:", X_val_style.shape)
print("Cache:", STYLOMETRY_PATH)
print("Stylometric feature extraction: PASSED")

Training stylometry:   0%|          | 0/217321 [00:00<?, ?it/s]

Validation stylometry:   0%|          | 0/55031 [00:00<?, ?it/s]

Extraction time: 8.49 minutes

Training features: (217321, 14)
Validation features: (55031, 14)
Cache: /content/drive/MyDrive/UCU_AI_HW3/experiment4_stylometric_features.npz
Stylometric feature extraction: PASSED


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

train_std = np.std(
    X_train_style.astype(np.float64),
    axis=0
)

val_std = np.std(
    X_val_style.astype(np.float64),
    axis=0
)

feature_summary = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "train_mean": X_train_style.mean(axis=0),
    "validation_mean": X_val_style.mean(axis=0),
    "train_std": train_std,
    "validation_std": val_std
})

display(feature_summary.round(6))

selected_mask = train_std > 1e-12

selected_features = [
    name
    for name, keep in zip(FEATURE_NAMES, selected_mask)
    if keep
]

removed_features = [
    name
    for name, keep in zip(FEATURE_NAMES, selected_mask)
    if not keep
]

X_train_style_selected = X_train_style[:, selected_mask]
X_val_style_selected = X_val_style[:, selected_mask]

assert X_train_style_selected.shape[1] > 0
assert X_train_style_selected.shape[1] == len(selected_features)

assert np.isfinite(X_train_style_selected).all()
assert np.isfinite(X_val_style_selected).all()

print("\nSelected features:", selected_features)
print("Removed features:", removed_features)

print("\nTraining shape:", X_train_style_selected.shape)
print("Validation shape:", X_val_style_selected.shape)

print("\nFeature statistics verification: PASSED")

,feature,train_mean,validation_mean,train_std,validation_std
0,log_char_count,7.824826,7.831393,0.896678,0.893839
1,log_word_count,6.010849,6.016578,0.854067,0.852356
2,mean_word_length,5.047474,5.052381,0.724491,0.718553
3,lexical_diversity,0.536993,0.536234,0.137893,0.137713
4,punctuation_ratio,0.027747,0.027633,0.013200,0.012897
5,digit_ratio,0.005297,0.005302,0.010573,0.010475
6,uppercase_ratio,0.028414,0.028270,0.020237,0.019708
7,newline_ratio,0.000000,0.000000,0.000000,0.000000
8,comma_per_word,0.053348,0.053179,0.022751,0.022592
9,period_per_word,0.058522,0.058389,0.041312,0.032860



Selected features: ['log_char_count', 'log_word_count', 'mean_word_length', 'lexical_diversity', 'punctuation_ratio', 'digit_ratio', 'uppercase_ratio', 'comma_per_word', 'period_per_word', 'exclamation_per_word', 'question_per_word', 'mean_sentence_words', 'sentence_length_std']
Removed features: ['newline_ratio']

Training shape: (217321, 13)
Validation shape: (55031, 13)

Feature statistics verification: PASSED



### 8.3. Training stylometric classifiers

I evaluate whether stylometric features improve the classification of human-written, AI-generated, and partially rewritten texts.

I train two Logistic Regression models:

1. A stylometry-only classifier using 13 numerical features.
2. A hybrid classifier combining word-level TF-IDF with standardized stylometric features.

I fit the feature scaler exclusively on the training folds and use the same fixed validation split as in the baseline and encoder experiments.

I compare macro-F1, class-specific F1, and accuracy. I also save the trained models and validation probabilities for subsequent ensembling experiments.


In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)
from IPython.display import display

SEED = 42
CLASS_ORDER = ["human", "ai", "fusion"]

y_train = baseline_train_df["label"].to_numpy()
y_val = baseline_val_df["label"].to_numpy()

style_scaler = StandardScaler()

X_train_style_scaled = style_scaler.fit_transform(
    X_train_style_selected
)

X_val_style_scaled = style_scaler.transform(
    X_val_style_selected
)

assert np.isfinite(X_train_style_scaled).all()
assert np.isfinite(X_val_style_scaled).all()

style_model = LogisticRegression(
    C=1.0,
    max_iter=500,
    solver="lbfgs",
    random_state=SEED
)

start_time = time.perf_counter()

style_model.fit(
    X_train_style_scaled,
    y_train
)

style_training_time = time.perf_counter() - start_time

style_val_preds = style_model.predict(
    X_val_style_scaled
)

style_val_probs = style_model.predict_proba(
    X_val_style_scaled
)

style_macro_f1 = f1_score(
    y_val,
    style_val_preds,
    labels=CLASS_ORDER,
    average="macro",
    zero_division=0
)

print("Stylometry-only macro-F1:", round(style_macro_f1, 5))
print("Training time:", round(style_training_time, 2), "seconds")

print("\nClassification report:")
print(
    classification_report(
        y_val,
        style_val_preds,
        labels=CLASS_ORDER,
        digits=4,
        zero_division=0
    )
)

Stylometry-only macro-F1: 0.6095
Training time: 6.56 seconds

Classification report:
              precision    recall  f1-score   support

       human     0.6056    0.4173    0.4941     13529
          ai     0.7602    0.8906    0.8203     24718
      fusion     0.5146    0.5137    0.5142     16784

    accuracy                         0.6593     55031
   macro avg     0.6268    0.6072    0.6095     55031
weighted avg     0.6473    0.6593    0.6467     55031



In [ ]:
from scipy.sparse import hstack, csr_matrix

assert "baseline_model" in globals()
assert "tfidf" in baseline_model.named_steps

word_vectorizer = baseline_model.named_steps["tfidf"]

X_train_tfidf = word_vectorizer.transform(
    baseline_train_df["text"]
)

X_val_tfidf = word_vectorizer.transform(
    baseline_val_df["text"]
)

STYLE_WEIGHT = 0.10

X_train_hybrid = hstack([
    X_train_tfidf,
    csr_matrix(
        X_train_style_scaled.astype(np.float32)
        * STYLE_WEIGHT
    )
], format="csr")

X_val_hybrid = hstack([
    X_val_tfidf,
    csr_matrix(
        X_val_style_scaled.astype(np.float32)
        * STYLE_WEIGHT
    )
], format="csr")

assert X_train_hybrid.shape[0] == 217_321
assert X_val_hybrid.shape[0] == 55_031

print("TF-IDF features:", X_train_tfidf.shape[1])
print("Stylometric features:", X_train_style_scaled.shape[1])
print("Combined features:", X_train_hybrid.shape[1])
print("Hybrid feature matrices: PASSED")

TF-IDF features: 150000
Stylometric features: 13
Combined features: 150013
Hybrid feature matrices: PASSED


In [ ]:
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

hybrid_model = LogisticRegression(
    C=2.0,
    max_iter=300,
    solver="lbfgs",
    random_state=SEED
)

start_time = time.perf_counter()

hybrid_model.fit(
    X_train_hybrid,
    y_train
)

hybrid_training_time = time.perf_counter() - start_time

hybrid_val_probs_raw = hybrid_model.predict_proba(
    X_val_hybrid
)

hybrid_class_indices = [
    list(hybrid_model.classes_).index(label)
    for label in CLASS_ORDER
]

hybrid_val_probs = hybrid_val_probs_raw[
    :, hybrid_class_indices
]

hybrid_val_preds = np.asarray(CLASS_ORDER)[
    np.argmax(hybrid_val_probs, axis=1)
]

hybrid_macro_f1 = f1_score(
    y_val,
    hybrid_val_preds,
    labels=CLASS_ORDER,
    average="macro",
    zero_division=0
)

print(f"Hybrid validation macro-F1: {hybrid_macro_f1:.5f}")
print(f"Training time: {hybrid_training_time / 60:.2f} minutes")

print("\nClassification report:")
print(
    classification_report(
        y_val,
        hybrid_val_preds,
        labels=CLASS_ORDER,
        digits=4,
        zero_division=0
    )
)

print("\nComparison:")
print("TF-IDF baseline:       0.87852")
print(f"TF-IDF + stylometry:   {hybrid_macro_f1:.5f}")
print(f"Improvement:           {hybrid_macro_f1 - 0.87852:+.5f}")

assert np.isfinite(hybrid_val_probs).all()
assert np.allclose(
    hybrid_val_probs.sum(axis=1),
    1.0,
    atol=1e-5
)

HYBRID_MODEL_PATH = (
    PROJECT_DIR / "experiment4_hybrid_model.joblib"
)

joblib.dump(
    {
        "classifier": hybrid_model,
        "scaler": style_scaler,
        "selected_features": selected_features,
        "style_weight": STYLE_WEIGHT,
        "class_order": CLASS_ORDER
    },
    HYBRID_MODEL_PATH
)

hybrid_probs_df = pd.DataFrame({
    "id": baseline_val_df["id"].astype(str).to_numpy(),
    "true_label": y_val,
    "hybrid_human": hybrid_val_probs[:, 0],
    "hybrid_ai": hybrid_val_probs[:, 1],
    "hybrid_fusion": hybrid_val_probs[:, 2]
})

HYBRID_PROBS_PATH = (
    PROJECT_DIR / "experiment4_hybrid_validation_probs.csv"
)

hybrid_probs_df.to_csv(
    HYBRID_PROBS_PATH,
    index=False
)

print("\nHybrid model saved:", HYBRID_MODEL_PATH)
print("Validation probabilities saved:", HYBRID_PROBS_PATH)
print("Experiment 4 training: PASSED")

Hybrid validation macro-F1: 0.88146
Training time: 6.74 minutes

Classification report:
              precision    recall  f1-score   support

       human     0.7814    0.8709    0.8238     13529
          ai     0.9842    0.9917    0.9879     24718
      fusion     0.8808    0.7896    0.8327     16784

    accuracy                         0.9004     55031
   macro avg     0.8821    0.8841    0.8815     55031
weighted avg     0.9028    0.9004    0.9002     55031


Comparison:
TF-IDF baseline:       0.87852
TF-IDF + stylometry:   0.88146
Improvement:           +0.00294

Hybrid model saved: /content/drive/MyDrive/UCU_AI_HW3/experiment4_hybrid_model.joblib
Validation probabilities saved: /content/drive/MyDrive/UCU_AI_HW3/experiment4_hybrid_validation_probs.csv
Experiment 4 training: PASSED



### 8.4. Three-model probability ensemble

I investigate whether the stylometry-enhanced TF-IDF classifier improves my existing 60/40 TF-IDF and XLM-RoBERTa ensemble.

I gradually replace part of the original TF-IDF contribution with probabilities from the hybrid classifier, while keeping the XLM-RoBERTa weight fixed at 40%.

I select the hybrid contribution using the group-aware calibration subset and evaluate the selected configuration on the held-out evaluation subset.

I compare the resulting model with the original 60/40 ensemble using macro-F1 and class-specific F1.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from IPython.display import display

CLASS_ORDER = ["human", "ai", "fusion"]

HYBRID_PROBS_PATH = (
    PROJECT_DIR / "experiment4_hybrid_validation_probs.csv"
)

hybrid_probs_df = pd.read_csv(
    HYBRID_PROBS_PATH,
    dtype={"id": str}
)

aligned_hybrid = ensemble_df[["id", "true_label"]].merge(
    hybrid_probs_df,
    on="id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_hybrid")
)

assert len(aligned_hybrid) == len(ensemble_df) == 55_031
assert aligned_hybrid["id"].is_unique

assert (
    aligned_hybrid["true_label"]
    == aligned_hybrid["true_label_hybrid"]
).all()

hybrid_probs = aligned_hybrid[
    ["hybrid_human", "hybrid_ai", "hybrid_fusion"]
].to_numpy(dtype=np.float64)

assert hybrid_probs.shape == (55_031, 3)
assert np.isfinite(hybrid_probs).all()
assert (hybrid_probs >= 0).all()
assert np.allclose(hybrid_probs.sum(axis=1), 1, atol=1e-4)

hybrid_predictions = np.asarray(CLASS_ORDER)[
    np.argmax(hybrid_probs, axis=1)
]

aligned_hybrid_f1 = f1_score(
    y_true,
    hybrid_predictions,
    labels=CLASS_ORDER,
    average="macro",
    zero_division=0
)

print(f"Aligned hybrid macro-F1: {aligned_hybrid_f1:.5f}")

assert abs(aligned_hybrid_f1 - 0.88146) < 0.0001

print("Three-model alignment: PASSED")

Aligned hybrid macro-F1: 0.88146
Three-model alignment: PASSED


In [ ]:
from sklearn.metrics import f1_score

def predict_three_model(indices, hybrid_weight):
    tfidf_weight = 0.60 - hybrid_weight

    combined = (
        tfidf_weight * tfidf_probs[indices]
        + 0.40 * xlmr_probs[indices]
        + hybrid_weight * hybrid_probs[indices]
    )

    return np.asarray(CLASS_ORDER)[
        np.argmax(combined, axis=1)
    ]

three_model_search_results = []

for hybrid_weight in np.round(
    np.linspace(0.0, 0.60, 13), 2
):
    predictions = predict_three_model(
        group_cal_idx,
        float(hybrid_weight)
    )

    macro_f1 = f1_score(
        y_true[group_cal_idx],
        predictions,
        labels=CLASS_ORDER,
        average="macro",
        zero_division=0
    )

    three_model_search_results.append({
        "tfidf_weight": float(0.60 - hybrid_weight),
        "xlmr_weight": 0.40,
        "hybrid_weight": float(hybrid_weight),
        "calibration_macro_f1": float(macro_f1)
    })

three_model_search_df = pd.DataFrame(
    three_model_search_results
).sort_values(
    ["calibration_macro_f1", "hybrid_weight"],
    ascending=[False, True]
).reset_index(drop=True)

best_hybrid_weight = float(
    three_model_search_df.iloc[0]["hybrid_weight"]
)

print("Best hybrid weight:", best_hybrid_weight)
print("TF-IDF weight:", round(0.60 - best_hybrid_weight, 2))
print("XLM-R weight: 0.40")

print("\nCalibration results:")
display(three_model_search_df.round(5))

three_model_search_df.to_csv(
    PROJECT_DIR / "experiment4_three_model_weight_search.csv",
    index=False
)

Best hybrid weight: 0.05
TF-IDF weight: 0.55
XLM-R weight: 0.40

Calibration results:


,tfidf_weight,xlmr_weight,hybrid_weight,calibration_macro_f1
0,0.55,0.4,0.05,0.91354
1,0.50,0.4,0.10,0.91330
2,0.45,0.4,0.15,0.91328
3,0.60,0.4,0.00,0.91325
4,0.30,0.4,0.30,0.91324
5,0.35,0.4,0.25,0.91322
6,0.10,0.4,0.50,0.91321
7,0.25,0.4,0.35,0.91321
8,0.40,0.4,0.20,0.91319
9,0.15,0.4,0.45,0.91319



### 8.4.3. Held-out evaluation of the three-model ensemble

I evaluate the selected three-model ensemble on the group-aware evaluation subset, which was not used to select the hybrid contribution.

I compare the original 60/40 TF-IDF and XLM-RoBERTa ensemble against the new 55/40/5 ensemble using macro-F1, accuracy, and class-specific F1.

I retain the three-model approach only if the additional stylometric contribution provides a meaningful and consistent improvement.


In [ ]:
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)

import numpy as np
import pandas as pd

evaluation_models = {
    "Fixed 60/40 ensemble": predict_three_model(
        group_eval_idx,
        hybrid_weight=0.0
    ),
    "Three-model ensemble": predict_three_model(
        group_eval_idx,
        hybrid_weight=best_hybrid_weight
    )
}

three_model_eval_rows = []

for name, predictions in evaluation_models.items():

    per_class_f1 = f1_score(
        y_true[group_eval_idx],
        predictions,
        labels=CLASS_ORDER,
        average=None,
        zero_division=0
    )

    three_model_eval_rows.append({
        "model": name,
        "macro_f1": float(per_class_f1.mean()),
        "accuracy": float(
            accuracy_score(y_true[group_eval_idx], predictions)
        ),
        "human_f1": float(per_class_f1[0]),
        "ai_f1": float(per_class_f1[1]),
        "fusion_f1": float(per_class_f1[2])
    })

three_model_eval_df = pd.DataFrame(three_model_eval_rows)

print("Held-out evaluation results:")
display(three_model_eval_df.round(5))

improvement = (
    three_model_eval_df.iloc[1]["macro_f1"]
    - three_model_eval_df.iloc[0]["macro_f1"]
)

print(f"\nMacro-F1 improvement: {improvement:+.5f}")

print("\nThree-model classification report:")
print(
    classification_report(
        y_true[group_eval_idx],
        evaluation_models["Three-model ensemble"],
        labels=CLASS_ORDER,
        digits=4,
        zero_division=0
    )
)

three_model_eval_df.to_csv(
    PROJECT_DIR / "experiment4_three_model_evaluation.csv",
    index=False
)

print("\nExperiment 4 evaluation: PASSED")

Held-out evaluation results:


,model,macro_f1,accuracy,human_f1,ai_f1,fusion_f1
0,Fixed 60/40 ensemble,0.9187,0.93253,0.87599,0.99320,0.88689
1,Three-model ensemble,0.9184,0.93228,0.87563,0.99317,0.88639



Macro-F1 improvement: -0.00030

Three-model classification report:
              precision    recall  f1-score   support

       human     0.8410    0.9132    0.8756      6750
          ai     0.9887    0.9977    0.9932     12526
      fusion     0.9265    0.8496    0.8864      8352

    accuracy                         0.9323     27628
   macro avg     0.9187    0.9202    0.9184     27628
weighted avg     0.9338    0.9323    0.9322     27628


Experiment 4 evaluation: PASSED



## 8. Final experimental findings

### Selected approach

My final approach combines word-level TF-IDF Logistic Regression (60%) with XLM-RoBERTa using head-tail inference (40%).

The ensemble achieved 0.91596 macro-F1 on the complete group-aware validation fold, compared with 0.87852 for TF-IDF and 0.87345 for XLM-RoBERTa.

On the group-aware evaluation subset, the fixed ensemble achieved 0.91870 macro-F1.

### Additional experiments

I investigated three extensions of the original ensemble:

- Class-specific decision calibration improved evaluation macro-F1 from 0.91870 to 0.91917.
- Length-aware ensemble weighting improved macro-F1 to 0.91935.
- Stylometric feature engineering improved the standalone TF-IDF classifier from 0.87852 to 0.88146, but the resulting three-model ensemble decreased evaluation macro-F1 from 0.91870 to 0.91840.

The additional ensemble improvements were small and were not considered sufficiently robust to replace the original 60/40 configuration.

### Model selection and limitations

I retained the original 60/40 ensemble as the final submission because it provided a substantial improvement over both individual models and had already been successfully evaluated on Kaggle.

The models were trained on the original training folds, while the ensemble weights were selected using validation data. Repeated experimentation on the same validation fold may lead to optimistic performance estimates.

All model comparisons use macro-F1, with additional class-specific analysis to identify differences in performance across human, AI-generated, and fusion texts.

### Final conclusion

Combining lexical TF-IDF features with contextual representations from XLM-RoBERTa yielded the strongest practically validated improvement in this notebook.

The experiments also demonstrated that adding complexity does not necessarily improve generalization: decision calibration, length-aware weighting, and stylometric features produced only marginal or inconsistent additional gains.
